## Week 3: Data Preprocessing

Team ds55 member: Yingxin Deng

This week tasks: 
1. Handle missing values (decide whether to drop, impute, or flag).
2. Convert categorical fields to numeric (encoding).
3. Normalize numerical features if needed.
4. Create train/test split: use the most recent month of available data as the test set, and the X months immediately preceding it as the training set. X is not fixed — treat the training window length as a tunable choice and experiment to determine the optimal value of X.
5. Deliverable: 02_preprocessing.ipynb + cleaned CSV.

In [1]:
import numpy as np
import pandas as pd

In [ ]:
files = [
    "../CRMLSSold202505.csv",
    "../CRMLSSold202506.csv",
    "../CRMLSSold202507.csv",
    "../CRMLSSold202508.csv",
    "../CRMLSSold202509.csv",
    "../CRMLSSold202510.csv",
    "../CRMLSSold202511.csv",
    "../CRMLSSold202512.csv",
    "../CRMLSSold202601.csv",
    "../CRMLSSold202602.csv",
    "../CRMLSSold202603.csv",
    "../CRMLSSold202604.csv",
    "../CRMLSSold202605.csv"
]

df = pd.concat([pd.read_csv(f) for f in files],ignore_index=True)
df = df[(df["PropertyType"] == "Residential") & (df["PropertySubType"] == "SingleFamilyResidence")]

missing_summary = pd.DataFrame({"missing_count": df.isnull().sum(),"missing_percent": df.isnull().mean() * 100})
missing_summary = missing_summary.sort_values("missing_percent", ascending=False)
missing_summary

/var/folders/tf/71q912hd3fzd4yz1qccgzpcw0000gn/T/ipykernel_765/1641812124.py:18: DtypeWarning: Columns (0: WaterfrontYN) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f) for f in files],
/var/folders/tf/71q912hd3fzd4yz1qccgzpcw0000gn/T/ipykernel_765/1641812124.py:18: DtypeWarning: Columns (0: WaterfrontYN, 1: PostalCode) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f) for f in files],


,missing_count,missing_percent
BusinessType,141997,100.0
CoveredSpaces,141997,100.0
MiddleOrJuniorSchoolDistrict,141997,100.0
FireplacesTotal,141997,100.0
TaxAnnualAmount,141997,100.0
...,...,...
StateOrProvince,0,0.0
ListingId,0,0.0
ListingContractDate,0,0.0
ContractStatusChangeDate,0,0.0


In [4]:
print("duplicates:")
duplicates = df["ListingKey"].duplicated().sum()
print(duplicates)

duplicates:
105


In [5]:
df = df.drop_duplicates(subset="ListingKey")

print("duplicates:")
duplicates = df["ListingKey"].duplicated().sum()
print(duplicates)

duplicates:
0


## Handle invalid data

In [6]:
df = df.copy()

# Invalid Latitude
before = df["Latitude"].isna().sum()
df.loc[(df["Latitude"] < 32) | (df["Latitude"] > 42), "Latitude"] = np.nan
after = df["Latitude"].isna().sum()
print(f"Latitude: {after - before} additional missing values.")

# Invalid Longitude
before = df["Longitude"].isna().sum()
df.loc[(df["Longitude"] < -125) | (df["Longitude"] > -114), "Longitude"] = np.nan
after = df["Longitude"].isna().sum()
print(f"Longitude: {after - before} additional missing values.")

# Invalid LotSizeSquareFeet
before = df["LotSizeSquareFeet"].isna().sum()
df.loc[df["LotSizeSquareFeet"] <= 100, "LotSizeSquareFeet"] = np.nan
after = df["LotSizeSquareFeet"].isna().sum()
print(f"LotSizeSquareFeet: {after - before} additional missing values.")

# Invalid LivingArea
before = df["LivingArea"].isna().sum()
df.loc[df["LivingArea"] <= 0, "LivingArea"] = np.nan
after = df["LivingArea"].isna().sum()
print(f"LivingArea: {after - before} additional missing values.")

# Invalid ParkingTotal
before = df["ParkingTotal"].isna().sum()
df.loc[df["ParkingTotal"] < 0, "ParkingTotal"] = np.nan
after = df["ParkingTotal"].isna().sum()
print(f"ParkingTotal: {after - before} additional missing values.")

# Invalid GarageSpaces
before = df["GarageSpaces"].isna().sum()
df.loc[df["GarageSpaces"] < 0, "GarageSpaces"] = np.nan
after = df["GarageSpaces"].isna().sum()
print(f"GarageSpaces: {after - before} additional missing values.")

# Invalid BathroomsTotalInteger
before = df["BathroomsTotalInteger"].isna().sum()
df.loc[df["BathroomsTotalInteger"] <= 0, "BathroomsTotalInteger"] = np.nan
after = df["BathroomsTotalInteger"].isna().sum()
print(f"BathroomsTotalInteger: {after - before} additional missing values.")

# Invalid BedroomsTotal
before = df["BedroomsTotal"].isna().sum()
df.loc[df["BedroomsTotal"] <= 0, "BedroomsTotal"] = np.nan
after = df["BedroomsTotal"].isna().sum()
print(f"BedroomsTotal: {after - before} additional missing values.")

# Invalid MainLevelBedrooms
before = df["MainLevelBedrooms"].isna().sum()
df.loc[df["MainLevelBedrooms"] < 0, "MainLevelBedrooms"] = np.nan
after = df["MainLevelBedrooms"].isna().sum()
print(f"MainLevelBedrooms: {after - before} additional missing values.")

# Invalid AssociationFee
before = df["AssociationFee"].isna().sum()
df.loc[df["AssociationFee"] < 0, "AssociationFee"] = np.nan
after = df["AssociationFee"].isna().sum()
print(f"AssociationFee: {after - before} additional missing values.")

# MainLevelBedrooms > BedroomsTotal
before = df["MainLevelBedrooms"].isna().sum()
df.loc[df["MainLevelBedrooms"] > df["BedroomsTotal"], "MainLevelBedrooms"] = np.nan
after = df["MainLevelBedrooms"].isna().sum()
print(f"MainLevelBedrooms consistency: {after - before} additional missing values.")

display(df.describe())

Latitude: 26 additional missing values.
Longitude: 40 additional missing values.
LotSizeSquareFeet: 361 additional missing values.
LivingArea: 63 additional missing values.
ParkingTotal: 20 additional missing values.
GarageSpaces: 0 additional missing values.
BathroomsTotalInteger: 61 additional missing values.
BedroomsTotal: 80 additional missing values.
MainLevelBedrooms: 0 additional missing values.
AssociationFee: 0 additional missing values.
MainLevelBedrooms consistency: 834 additional missing values.


,OriginalListPrice,ListingKey,ClosePrice,Latitude,Longitude,LivingArea,ListPrice,DaysOnMarket,FireplacesTotal,AboveGradeFinishedArea,...,ElementarySchoolDistrict,BelowGradeFinishedArea,CoveredSpaces,Stories,LotSizeArea,MainLevelBedrooms,GarageSpaces,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict
count,1.415960e+05,1.418920e+05,1.418920e+05,141853.000000,141839.000000,141751.000000,1.418920e+05,141892.000000,0.0,0.0,...,0.0,1019.000000,0.0,127122.000000,1.394740e+05,85493.000000,136276.000000,100536.000000,1.391000e+05,0.0
mean,1.384827e+06,1.130129e+09,1.345097e+06,34.738173,-118.637365,2049.783373,1.267135e+06,39.832443,NaN,NaN,...,NaN,65.338567,NaN,1.352268,1.835489e+04,2.241435,2.006727,106.487993,3.722827e+05,NaN
std,7.945244e+06,1.872800e+07,8.087425e+06,1.696487,1.844924,1043.617009,1.602429e+06,52.923667,NaN,NaN,...,NaN,288.546592,NaN,0.477679,2.013758e+05,1.359447,3.306508,345.452913,1.747708e+07,NaN
min,0.000000e+00,4.217759e+08,0.000000e+00,32.545249,-124.193201,100.000000,8.000000e+03,-265.000000,NaN,NaN,...,NaN,0.000000,NaN,1.000000,0.000000e+00,0.000000,0.000000,0.000000,1.120000e+02,NaN
25%,6.389675e+05,1.114251e+09,6.250000e+05,33.761504,-119.148654,1387.000000,6.250000e+05,8.000000,NaN,NaN,...,NaN,0.000000,NaN,1.000000,5.413000e+03,1.000000,2.000000,0.000000,5.663000e+03,NaN
50%,8.990000e+05,1.127488e+09,8.900000e+05,34.083408,-118.031504,1820.000000,8.950000e+05,20.000000,NaN,NaN,...,NaN,0.000000,NaN,1.000000,7.100000e+03,3.000000,2.000000,0.000000,7.300000e+03,NaN
75%,1.449000e+06,1.149381e+09,1.425000e+06,34.816486,-117.260317,2441.000000,1.399912e+06,52.000000,NaN,NaN,...,NaN,0.000000,NaN,2.000000,1.000000e+04,3.000000,2.000000,130.000000,1.045400e+04,NaN
max,1.302000e+09,1.171669e+09,9.895000e+08,41.894714,-114.347243,56500.000000,1.375000e+08,2177.000000,NaN,NaN,...,NaN,3490.000000,NaN,2.000000,4.379087e+07,16.000000,600.000000,20712.000000,1.938943e+09,NaN


In [7]:
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": df.isnull().mean() * 100
})

missing_summary = missing_summary.sort_values("missing_percent", ascending=False)
missing_summary

,missing_count,missing_percent
BusinessType,141892,100.0
CoveredSpaces,141892,100.0
MiddleOrJuniorSchoolDistrict,141892,100.0
FireplacesTotal,141892,100.0
TaxAnnualAmount,141892,100.0
...,...,...
MlsStatus,0,0.0
StateOrProvince,0,0.0
ListingContractDate,0,0.0
ContractStatusChangeDate,0,0.0


## Handle Missing Values

Remove features with excessive missing values

Columns with more than 70% missing values were removed. Since fewer than 30% of observations contain valid information for these features, they provide little predictive value while increasing data sparsity and preprocessing complexity.

In [8]:
missing_percent = df.isnull().mean() * 100
drop_cols = missing_percent[missing_percent > 70].index.tolist()

print(f"Number of columns to drop: {len(drop_cols)}")
print(drop_cols)

# Drop those columns
df = df.drop(columns=drop_cols)

print(f"Remaining columns: {df.shape[1]}")

Number of columns to drop: 22
['WaterfrontYN', 'BasementYN', 'CoListOfficeName', 'CoListAgentFirstName', 'CoListAgentLastName', 'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea', 'TaxAnnualAmount', 'ElementarySchool', 'BuilderName', 'TaxYear', 'BuildingAreaTotal', 'ElementarySchoolDistrict', 'CoBuyerAgentFirstName', 'BelowGradeFinishedArea', 'BusinessType', 'CoveredSpaces', 'MiddleOrJuniorSchool', 'HighSchool', 'LotSizeDimensions', 'MiddleOrJuniorSchoolDistrict']
Remaining columns: 56


In addition, columns containing personal names or agent information (e.g., first names, last names, and office names) were removed. These features are not directly related to property characteristics and are unlikely to contribute meaningfully to house price prediction. They also have high cardinality, which would unnecessarily increase the complexity of feature encoding without providing significant predictive benefits.

In [9]:
object_cols = df.select_dtypes(include=["object"]).columns.tolist()

for col in object_cols:
    print(col)

BuyerAgentAOR
ListAgentAOR
Flooring
ViewYN
PoolPrivateYN
ListAgentEmail
CloseDate
ListAgentFirstName
ListAgentLastName
UnparsedAddress
PropertyType
ListOfficeName
BuyerOfficeName
ListAgentFullName
BuyerAgentMlsId
BuyerAgentFirstName
BuyerAgentLastName
MLSAreaMajor
CountyOrParish
MlsStatus
AttachedGarageYN
PropertySubType
SubdivisionName
BuyerOfficeAOR
ListingId
City
ContractStatusChangeDate
PurchaseContractDate
ListingContractDate
StateOrProvince
FireplaceYN
Levels
NewConstructionYN
HighSchoolDistrict
PostalCode


/var/folders/tf/71q912hd3fzd4yz1qccgzpcw0000gn/T/ipykernel_765/3861773157.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df.select_dtypes(include=["object"]).columns.tolist()


In [10]:
personal_cols = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in [
        "agent",
        "office",
        "email",
        "mlsid",
        "name"
    ])
]

print(personal_cols)
df = df.drop(columns=personal_cols)
print(f"Remaining columns: {df.shape[1]}")

['BuyerAgentAOR', 'ListAgentAOR', 'ListAgentEmail', 'ListAgentFirstName', 'ListAgentLastName', 'ListOfficeName', 'BuyerOfficeName', 'ListAgentFullName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'SubdivisionName', 'BuyerOfficeAOR']
Remaining columns: 43


In [11]:
# Keep LotSizeSquareFeet and remove redundant lot size features
drop_lot_cols = [
    "LotSizeAcres",
    "LotSizeArea"
]

df = df.drop(columns=[col for col in drop_lot_cols if col in df.columns])

print("Using LotSizeSquareFeet as the lot size feature.")
print("Dropped:", drop_lot_cols)

Using LotSizeSquareFeet as the lot size feature.
Dropped: ['LotSizeAcres', 'LotSizeArea']


ListPrice and OriginalListPrice were also removed to avoid data leakage, since they are highly related to the final sale price.

In [12]:
LEAKAGE_COLUMNS = [
    "ListPrice",
    "OriginalListPrice"
]

df = df.drop(columns=LEAKAGE_COLUMNS)

In [13]:
df["CloseDate"] = pd.to_datetime(df["CloseDate"])

In [14]:
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": df.isnull().mean() * 100
})

missing_summary = (
    missing_summary[missing_summary["missing_count"] > 0]
    .sort_values("missing_percent", ascending=False)
)

missing_summary

,missing_count,missing_percent
MainLevelBedrooms,56399,39.747836
Flooring,50485,35.579878
AssociationFee,41356,29.146111
HighSchoolDistrict,38266,26.968399
MLSAreaMajor,20803,14.661151
AttachedGarageYN,17183,12.109915
Stories,14770,10.409325
ViewYN,12899,9.090717
PoolPrivateYN,10976,7.735461
NewConstructionYN,10829,7.631861


Missing values handling rules：

1. Columns such as StreetNumberNumeric and UnparsedAddress were removed because detailed address information is not directly useful after keeping broader location variables such as city, postal code, latitude, and longitude. 

2. For categorical variables such as Flooring, HighSchoolDistrict, MLSAreaMajor, Levels, City, and PostalCode, missing values were filled with "Unknown" because the absence of information should be treated as a separate category rather than replaced with the most common value. 

3. For binary Yes/No variables such as AttachedGarageYN, ViewYN, PoolPrivateYN, NewConstructionYN, and FireplaceYN, missing values were filled with "N" and then encoded as 0/1, because missing entries in MLS-style property records often indicate that the feature was not present or not reported.

4. For numerical variables such as AssociationFee, Stories, GarageSpaces, LotSizeSquareFeet, YearBuilt, LivingArea, BathroomsTotalInteger, and ParkingTotal, missing values were imputed using the median. The median was chosen because real estate variables are often skewed and can contain extreme outliers. 

5. MainLevelBedrooms was handled separately and missing values were filled with 0, since a missing value may indicate that there are no bedrooms on the main level. 

6. Finally, rows with missing Latitude, Longitude, or PurchaseContractDate were dropped because only a very small number of records were affected, and these fields are important for location- and time-based modeling.

## Train-Test Split

Since the project requires the training window length to be tunable, I started with a six-month training window as a reasonable baseline. Different values of X (e.g., 3, 6, 9, and 12 months) will be evaluated in the modeling stage to determine which window provides the best predictive performance.

In [15]:
df["CloseDate"] = pd.to_datetime(df["CloseDate"])
df = df.sort_values("CloseDate").reset_index(drop=True)
df["YearMonth"] = df["CloseDate"].dt.to_period("M")

print("Available months:")
print(df["YearMonth"].value_counts().sort_index())

latest_month = df["YearMonth"].max()

# Initial training window length
train_window = 6

# Get the X months immediately before the test month
train_months = pd.period_range(
    end=latest_month - 1,
    periods=train_window,
    freq="M"
)

train_df = df[df["YearMonth"].isin(train_months)].copy()
test_df = df[df["YearMonth"] == latest_month].copy()

print("\nLatest month / Test month:", latest_month)

print("\nTraining months:")
print(sorted(train_df["YearMonth"].unique()))

print("\nTest month:")
print(sorted(test_df["YearMonth"].unique()))

print("\nTrain shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df = train_df.drop(columns=["YearMonth"])
test_df = test_df.drop(columns=["YearMonth"])

Available months:
YearMonth
2025-05    11776
2025-06    11689
2025-07    12104
2025-08    11448
2025-09    11448
2025-10    12024
2025-11     9732
2025-12    10439
2026-01     7481
2026-02     8544
2026-03    11166
2026-04    12024
2026-05    12017
Freq: M, Name: count, dtype: int64

Latest month / Test month: 2026-05

Training months:
[Period('2025-11', 'M'), Period('2025-12', 'M'), Period('2026-01', 'M'), Period('2026-02', 'M'), Period('2026-03', 'M'), Period('2026-04', 'M')]

Test month:
[Period('2026-05', 'M')]

Train shape: (59386, 40)
Test shape: (12017, 40)


In [16]:
# Separate features and target
X_train = train_df.drop(columns=["ClosePrice"])
y_train = train_df["ClosePrice"]

X_test = test_df.drop(columns=["ClosePrice"])
y_test = test_df["ClosePrice"]

Handle missing values after train/test split

In [18]:
# Drop columns that should not be used for modeling
drop_cols = [
    "StreetNumberNumeric",
    "UnparsedAddress",
    "BedroomsTotal" #duplicate with bedroomsTotalInteger
]

train_df = train_df.drop(columns=[col for col in drop_cols if col in train_df.columns])
test_df = test_df.drop(columns=[col for col in drop_cols if col in test_df.columns])


# Handle MainLevelBedrooms separately
for data in [train_df, test_df]:
    if "MainLevelBedrooms" in data.columns:
        data["MainLevelBedrooms"] = data["MainLevelBedrooms"].fillna(0)


# Fill categorical variables with "Unknown"
categorical_unknown_cols = [
    "Flooring",
    "HighSchoolDistrict",
    "MLSAreaMajor",
    "Levels",
    "City",
    "PostalCode"
]

for data in [train_df, test_df]:
    for col in categorical_unknown_cols:
        if col in data.columns:
            data[col] = data[col].fillna("Unknown")


# Fill Yes/No variables with "N" and encode them as 0/1
yn_cols = [
    "AttachedGarageYN",
    "ViewYN",
    "PoolPrivateYN",
    "NewConstructionYN",
    "FireplaceYN"
]

yn_mapping = {
    "Y": 1,
    "N": 0,
    True: 1,
    False: 0,
    "True": 1,
    "False": 0
}

for data in [train_df, test_df]:
    for col in yn_cols:
        if col in data.columns:
            data[col] = data[col].fillna("N").map(yn_mapping)


# Fill numerical variables with TRAIN median to avoid data leakage
median_cols = [
    "AssociationFee",
    "Stories",
    "GarageSpaces",
    "LotSizeSquareFeet",
    "YearBuilt",
    "LivingArea",
    "BathroomsTotalInteger",
    "ParkingTotal"
]

for col in median_cols:
    if col in train_df.columns and col in test_df.columns:
        train_median = train_df[col].median()

        train_df[col] = train_df[col].fillna(train_median)
        test_df[col] = test_df[col].fillna(train_median)


# Drop rows with missing values in important location/date fields
dropna_subset = [
    "Latitude",
    "Longitude",
    "PurchaseContractDate"
]

dropna_subset_train = [col for col in dropna_subset if col in train_df.columns]
dropna_subset_test = [col for col in dropna_subset if col in test_df.columns]

train_df = train_df.dropna(subset=dropna_subset_train)
test_df = test_df.dropna(subset=dropna_subset_test)


# Final missing value check
train_missing = train_df.isnull().sum()
train_missing = train_missing[train_missing > 0].sort_values(ascending=False)

test_missing = test_df.isnull().sum()
test_missing = test_missing[test_missing > 0].sort_values(ascending=False)

print("Remaining missing values in train:")
print(train_missing)

print("\nRemaining missing values in test:")
print(test_missing)

Remaining missing values in train:
Series([], dtype: int64)

Remaining missing values in test:
Series([], dtype: int64)


## Feature Encoding

In [19]:
df.select_dtypes(include=["object"]).columns.tolist()

/var/folders/tf/71q912hd3fzd4yz1qccgzpcw0000gn/T/ipykernel_765/392840347.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include=["object"]).columns.tolist()


['Flooring',
 'ViewYN',
 'PoolPrivateYN',
 'UnparsedAddress',
 'PropertyType',
 'MLSAreaMajor',
 'CountyOrParish',
 'MlsStatus',
 'AttachedGarageYN',
 'PropertySubType',
 'ListingId',
 'City',
 'ContractStatusChangeDate',
 'PurchaseContractDate',
 'ListingContractDate',
 'StateOrProvince',
 'FireplaceYN',
 'Levels',
 'NewConstructionYN',
 'HighSchoolDistrict',
 'PostalCode']

In [20]:
categorical_cols = train_df.select_dtypes(include=["object"]).columns

unique_summary = pd.DataFrame({
    "Unique Values": train_df[categorical_cols].nunique()
}).sort_values("Unique Values", ascending=False)

unique_summary

/var/folders/tf/71q912hd3fzd4yz1qccgzpcw0000gn/T/ipykernel_765/1719114483.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = train_df.select_dtypes(include=["object"]).columns


,Unique Values
ListingId,59354
PostalCode,1601
MLSAreaMajor,969
City,893
ListingContractDate,688
HighSchoolDistrict,398
PurchaseContractDate,390
Flooring,225
ContractStatusChangeDate,179
CountyOrParish,59


In [44]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# --------------------------------------------------
# 1. Drop unused columns
# --------------------------------------------------

drop_before_encoding = [
    "ListingId",
    "ListingKey",
    "ContractStatusChangeDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "PropertySubType",       # SingleFamilyResidence
    "PropertyType",          # Residential
    "MlsStatus",
    "StateOrProvince",       # All data are from CA
    "Levels",                # Overlaps with Stories
    "CountyOrParish",        # MLSAreaMajor is used for location
    "City",
    "PostalCode",
    "HighSchoolDistrict",
    "Flooring"
]

train_df = train_df.drop(
    columns=[
        col for col in drop_before_encoding
        if col in train_df.columns
    ]
)

test_df = test_df.drop(
    columns=[
        col for col in drop_before_encoding
        if col in test_df.columns
    ]
)


# --------------------------------------------------
# 2. One-hot encode MLSAreaMajor
# --------------------------------------------------

encoder = OneHotEncoder(
    drop="first",
    handle_unknown="ignore",
    sparse_output=False
)

encoded_train_array = encoder.fit_transform(
    train_df[["MLSAreaMajor"]]
)

encoded_test_array = encoder.transform(
    test_df[["MLSAreaMajor"]]
)

encoded_cols = encoder.get_feature_names_out(
    ["MLSAreaMajor"]
)


# --------------------------------------------------
# 3. Convert encoded arrays to DataFrames
# --------------------------------------------------

encoded_train_df = pd.DataFrame(
    encoded_train_array,
    columns=encoded_cols,
    index=train_df.index
)

encoded_test_df = pd.DataFrame(
    encoded_test_array,
    columns=encoded_cols,
    index=test_df.index
)


# --------------------------------------------------
# 4. Replace original MLSAreaMajor with encoded columns
# --------------------------------------------------

X_train = pd.concat(
    [
        train_df.drop(columns=["MLSAreaMajor"]),
        encoded_train_df
    ],
    axis=1
)

X_test = pd.concat(
    [
        test_df.drop(columns=["MLSAreaMajor"]),
        encoded_test_df
    ],
    axis=1
)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


## quality check

In [45]:
print("=" * 60)
print("Data preprocessing completed.")
print("=" * 60)

print("\nShapes before encoding:")
print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")

print("\nShapes after encoding:")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

print(f"\nNumber of MLSAreaMajor dummy columns: {len(encoded_cols)}")

print(
    f"\nRemaining missing values in X_train: "
    f"{X_train.isna().sum().sum()}"
)

print(
    f"Remaining missing values in X_test: "
    f"{X_test.isna().sum().sum()}"
)

print(
    f"\nObject columns in X_train: "
    f"{len(X_train.select_dtypes(include='object').columns)}"
)

print(
    f"Object columns in X_test: "
    f"{len(X_test.select_dtypes(include='object').columns)}"
)

print(
    f"\nTrain/Test columns identical: "
    f"{X_train.columns.equals(X_test.columns)}"
)

print(
    f"Original MLSAreaMajor removed: "
    f"{'MLSAreaMajor' not in X_train.columns}"
)

print("\nFirst 20 final features:")
print(X_train.columns[:20].tolist())

print(f"\nTotal number of final features: {X_train.shape[1]}")

Data preprocessing completed.

Shapes before encoding:
Train shape: (59354, 22)
Test shape:  (12009, 22)

Shapes after encoding:
X_train shape: (59354, 989)
X_test shape:  (12009, 989)

Number of MLSAreaMajor dummy columns: 968

Remaining missing values in X_train: 0
Remaining missing values in X_test: 0

Object columns in X_train: 1
Object columns in X_test: 1

Train/Test columns identical: True
Original MLSAreaMajor removed: True

First 20 final features:
['ViewYN', 'PoolPrivateYN', 'CloseDate', 'ClosePrice', 'Latitude', 'Longitude', 'LivingArea', 'DaysOnMarket', 'ListingKeyNumeric', 'AttachedGarageYN', 'ParkingTotal', 'YearBuilt', 'BathroomsTotalInteger', 'FireplaceYN', 'Stories', 'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces', 'AssociationFee', 'LotSizeSquareFeet']

Total number of final features: 989


/var/folders/tf/71q912hd3fzd4yz1qccgzpcw0000gn/T/ipykernel_765/1558136385.py:27: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  f"{len(X_train.select_dtypes(include='object').columns)}"
/var/folders/tf/71q912hd3fzd4yz1qccgzpcw0000gn/T/ipykernel_765/1558136385.py:32: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pan

## Save Week 3 deliverables

In [46]:
# Add a column indicating the dataset split
train_df["Dataset"] = "Train"
test_df["Dataset"] = "Test"

# Combine the datasets
cleaned_df = pd.concat([train_df, test_df], ignore_index=True)

# Optional: sort by CloseDate if you want to preserve chronological order
cleaned_df = cleaned_df.sort_values("CloseDate").reset_index(drop=True)

# Save the cleaned dataset
cleaned_df.to_csv(
    "../week3/cleaned_data_before_one_hot_encoding.csv",
    index=False
)

print(cleaned_df["Dataset"].value_counts())

Dataset
Train    59354
Test     12009
Name: count, dtype: int64


In [47]:
# Reconstruct processed train and test datasets
train_processed = X_train.copy()
train_processed["ClosePrice"] = y_train
train_processed["Dataset"] = "Train"

test_processed = X_test.copy()
test_processed["ClosePrice"] = y_test
test_processed["Dataset"] = "Test"

# Combine train and test
cleaned_df = pd.concat(
    [train_processed, test_processed],
    ignore_index=True
)

# Save processed dataset
cleaned_df.to_csv(
    "../week3/cleaned_data.csv.gz",
    index=False
)

print("=" * 50)
print("Processed dataset saved successfully.")
print("=" * 50)

print(cleaned_df["Dataset"].value_counts())

print(f"\nFinal dataset shape: {cleaned_df.shape}")

Processed dataset saved successfully.
Dataset
Train    59354
Test     12009
Name: count, dtype: int64

Final dataset shape: (71363, 989)
